In [1]:
import os

# SSL certificate configuration:
# Uses the Linux system's trusted CA certificates for HTTPS connections.
# This is required in our corporate network because the default Python
# certificate bundle does not trust the network's certificate chain.
os.environ['REQUESTS_CA_BUNDLE'] = '/etc/ssl/certs/ca-certificates.crt'
os.environ['SSL_CERT_FILE'] = '/etc/ssl/certs/ca-certificates.crt'

# Mini Semantic Search Engine

## Objective

Implement a semantic search engine in Python using Sentence Transformers and FAISS.

The system converts a domain-specific knowledge base into vector embeddings, stores the embeddings in a FAISS index, and retrieves the most semantically relevant information for a given user query.

### Key Components

- Sentence Transformers for text embedding generation
- `all-MiniLM-L6-v2` for 384-dimensional embeddings
- FAISS for vector indexing and similarity search
- L2 normalization for cosine-similarity-based retrieval
- Interactive CLI for continuous semantic search

### Project Scope

The implementation covers the complete retrieval workflow:

1. Knowledge base creation
2. Embedding generation
3. Vector normalization
4. FAISS index construction
5. Semantic similarity search
6. Top-3 result retrieval
7. Interactive query interface

In [2]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

/home/nineleaps/Documents/da_python/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load the Embedding Model

`all-MiniLM-L6-v2` is a lightweight Sentence Transformer model designed to generate semantic representations of text.

Each input sentence is converted into a vector containing 384 numerical dimensions. These vectors can then be compared based on their mathematical similarity.

In [3]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4654.81it/s]


## 3. Create the Knowledge Base

The knowledge base represents support information for a food delivery platform.

The dataset contains information related to order tracking, delivery delays, missing items, refunds, addresses, promotions, restaurants, drivers, and estimated delivery times.

The sentences are intentionally written in natural language so that semantic search can be evaluated using queries that may express the same intent with different wording.

In [4]:
knowledge_base = [
    'Customers can track their food order in real time from the order tracking page.',
    'Orders may be delayed during periods of heavy traffic or unusually high demand.',
    'Customers should contact support if their delivered order contains missing items.',
    'A refund may be issued when an order is cancelled after payment has been completed.',
    'Customers can change the delivery address before the restaurant starts preparing the order.',
    'Promotional coupons may have restrictions such as minimum order values or expiration dates.',
    'Customers can report incorrect food items through the order details and support section.',
    'Restaurants may temporarily stop accepting orders when they reach their maximum capacity.',
    'The delivery driver can contact the customer if they are unable to locate the delivery address.',
    'Customers can view the estimated delivery time before placing an order.'
]

## 4. Generate Embeddings

The knowledge-base sentences are passed to the Sentence Transformer model to generate their corresponding vector representations.

With 10 sentences and a 384-dimensional embedding for each sentence, the resulting embedding matrix is expected to have the shape `(10, 384)`.

In [5]:
embeddings = model.encode(knowledge_base)

print('Embedding shape:', embeddings.shape)

Embedding shape: (10, 384)


## 5. Build the FAISS Index

FAISS is used to store and search the generated embedding vectors efficiently.

The `IndexFlatL2` index calculates the L2 (Euclidean) distance between vectors and performs an exact nearest-neighbour search.

The embedding vectors are normalized before being added to the index. Normalization allows L2 distance between the normalized vectors to be used as a measure equivalent to cosine similarity for ranking purposes.

In [6]:
embedding_dim = 384

index = faiss.IndexFlatL2(embedding_dim)

print('FAISS index created successfully.')
print('Index dimension:', index.d)
print('Vectors stored:', index.ntotal)

FAISS index created successfully.
Index dimension: 384
Vectors stored: 0


## 6. Normalize the Embeddings

The generated embeddings are normalized to unit length before being added to FAISS.

Normalization scales each embedding so that its magnitude becomes 1. For two unit-length vectors, L2 distance and cosine similarity produce the same ranking of vectors.

This allows `IndexFlatL2` to be used for cosine-similarity-based semantic retrieval.

In [7]:
faiss.normalize_L2(embeddings)

print('Embeddings normalized successfully.')
print('Embedding shape:', embeddings.shape)

Embeddings normalized successfully.
Embedding shape: (10, 384)


## 5. Build the FAISS Index

FAISS is used to store and search the generated embedding vectors efficiently.

The `IndexFlatL2` index calculates the L2 (Euclidean) distance between vectors and performs an exact nearest-neighbour search.

The embedding vectors are normalized before being added to the index. Normalization allows L2 distance between the normalized vectors to be used as a measure equivalent to cosine similarity for ranking purposes.

In [8]:
embedding_dim = 384

index = faiss.IndexFlatL2(embedding_dim)

print('FAISS index created successfully.')
print('Index dimension:', index.d)
print('Vectors stored:', index.ntotal)

FAISS index created successfully.
Index dimension: 384
Vectors stored: 0


## 6. Normalize the Embeddings

The generated embeddings are normalized to unit length before being added to FAISS.

Normalization scales each embedding so that its magnitude becomes 1. For two unit-length vectors, L2 distance and cosine similarity produce the same ranking of vectors.

This allows `IndexFlatL2` to be used for cosine-similarity-based semantic retrieval.

In [9]:
faiss.normalize_L2(embeddings)

print('Embeddings normalized successfully.')
print('Embedding shape:', embeddings.shape)

Embeddings normalized successfully.
Embedding shape: (10, 384)


## 7. Add Embeddings to the FAISS Index

The normalized knowledge-base embeddings are added to the FAISS index.

Each row in the embedding matrix represents one knowledge-base sentence, so the index should contain 10 vectors after insertion.

In [10]:
index.add(embeddings)

print('Embeddings added to FAISS.')
print('Total vectors stored:', index.ntotal)

Embeddings added to FAISS.
Total vectors stored: 10


## 8. Semantic Search

A user query is converted into the same 384-dimensional embedding space as the knowledge-base sentences.

The query vector is normalized before searching the FAISS index. The normalized query is then compared against the normalized knowledge-base vectors.

The search returns the three nearest vectors along with their L2 distances.

Since the vectors are normalized, smaller L2 distance indicates greater semantic similarity.

In [11]:
query = 'Where can I see the current location of my food order?'

query_embedding = model.encode([query])
faiss.normalize_L2(query_embedding)

print('Query embedding shape:', query_embedding.shape)

Query embedding shape: (1, 384)


## 9. Retrieve the Top 3 Results

The FAISS `search()` method compares the query vector against the vectors stored in the index.

The parameter `k=3` requests the three nearest vectors.

The search returns two arrays:

- `distances` — the L2 distance between the query and each matched vector
- `indices` — the positions of the matched vectors in the original knowledge base

The results are ordered from the smallest distance to the largest distance.

In [12]:
distances, indices = index.search(query_embedding, k=3)

print('Distances:', distances)
print('Indices:', indices)

Distances: [[0.7110611 1.0218267 1.1433561]]
Indices: [[0 6 4]]


## 10. Display Search Results

The FAISS search results are mapped back to their corresponding knowledge-base sentences.

Results are displayed in ranked order, with the rank, similarity score, and matched sentence.

For `IndexFlatL2`, the returned score represents L2 distance. Because the embeddings were normalized, a lower distance indicates greater similarity.

In [13]:
print(f'{"Rank":<6} {"Score":<12} Matched Sentence')
print('-' * 100)

for rank, (distance, idx) in enumerate(zip(distances[0], indices[0]), start=1):
    print(f'{rank:<6} {distance:<12.4f} {knowledge_base[idx]}')

Rank   Score        Matched Sentence
----------------------------------------------------------------------------------------------------
1      0.7111       Customers can track their food order in real time from the order tracking page.
2      1.0218       Customers can report incorrect food items through the order details and support section.
3      1.1434       Customers can change the delivery address before the restaurant starts preparing the order.


## 11. Semantic Search Test — Query 2

A second query is used to evaluate whether the search engine can identify relevant information when the query wording differs from the wording used in the knowledge base.

In [14]:
query = 'My delivered meal is missing some items. What should I do?'

query_embedding = model.encode([query])
faiss.normalize_L2(query_embedding)

distances, indices = index.search(query_embedding, k=3)

print(f'{"Rank":<6} {"Score":<12} Matched Sentence')
print('-' * 100)

for rank, (distance, idx) in enumerate(zip(distances[0], indices[0]), start=1):
    print(f'{rank:<6} {distance:<12.4f} {knowledge_base[idx]}')

Rank   Score        Matched Sentence
----------------------------------------------------------------------------------------------------
1      0.7257       Customers should contact support if their delivered order contains missing items.
2      0.9578       Customers can report incorrect food items through the order details and support section.
3      1.2962       Customers can track their food order in real time from the order tracking page.


## 12. Semantic Search Test — Query 3

A third query tests retrieval for a payment and cancellation-related scenario using wording that differs from the corresponding knowledge-base sentence.

In [15]:
query = 'The restaurant cancelled my order after I paid. Can I get my money back?'

query_embedding = model.encode([query])
faiss.normalize_L2(query_embedding)

distances, indices = index.search(query_embedding, k=3)

print(f'{"Rank":<6} {"Score":<12} Matched Sentence')
print('-' * 100)

for rank, (distance, idx) in enumerate(zip(distances[0], indices[0]), start=1):
    print(f'{rank:<6} {distance:<12.4f} {knowledge_base[idx]}')

Rank   Score        Matched Sentence
----------------------------------------------------------------------------------------------------
1      0.7339       A refund may be issued when an order is cancelled after payment has been completed.
2      1.0607       Restaurants may temporarily stop accepting orders when they reach their maximum capacity.
3      1.0918       Customers should contact support if their delivered order contains missing items.


## 13. Interactive Semantic Search

An interactive command-line interface allows users to submit multiple queries without modifying the underlying code.

For every query, the system:

1. Converts the query into an embedding.
2. Normalizes the query vector.
3. Searches the FAISS index for the three nearest vectors.
4. Retrieves the corresponding knowledge-base sentences.
5. Displays the ranked results.

The interface continues running until the user enters `exit`.

In [16]:
while True:
    query = input('\nEnter your query (type "exit" to quit): ')

    if query.lower() == 'exit':
        print('Exiting semantic search.')
        break

    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)

    distances, indices = index.search(query_embedding, k=3)

    print(f'\n{"Rank":<6} {"Score":<12} Matched Sentence')
    print('-' * 100)

    for rank, (distance, idx) in enumerate(
        zip(distances[0], indices[0]), start=1
    ):
        print(f'{rank:<6} {distance:<12.4f} {knowledge_base[idx]}')


Rank   Score        Matched Sentence
----------------------------------------------------------------------------------------------------
1      1.0296       Customers should contact support if their delivered order contains missing items.
2      1.1865       Customers can report incorrect food items through the order details and support section.
3      1.2440       Customers can track their food order in real time from the order tracking page.
Exiting semantic search.
